# Observability - See Everything Your AI Does

**Module 7 · Lesson 7.6**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Langfuse - Open-Source LLM Observability
- LangSmith - The LangChain Ecosystem Tool
- Quality Scoring - Was the Answer Actually Good?
- Dashboards - The Metrics That Matter
- Tool Landscape - Helicone, Phoenix, Portkey, OpenTelemetry
- Advanced Langfuse - Prompts, Datasets, Experiments, Annotation Queues

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install openai "langfuse>=3" langsmith google-genai python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


## The Outage Nobody Could Explain

A team's AI support bot started giving worse answers on a Tuesday. Users complained. The on-call engineer opened the dashboards:

**📝 `incident.txt`** - *Intro*

In [ ]:
# Output: the incident channel, reconstructed
#
# 14:02  "answers are wrong on billing questions"   (3 users)
# 14:20  on-call checks: latency NORMAL, error rate NORMAL, uptime 100%
# 14:41  "still bad" (12 users). on-call: "everything is green?!"
# 15:30  guessing. rolls back a deploy. no change. rolls it forward. no change.
# 18:10  someone asks: "what PROMPT are we actually sending now?"
#        ...nobody knows. there is no record of any single request.
#
# Root cause (found next day, by luck): the provider silently updated the
# model behind the same name. Quality dropped. Latency and errors never moved.
# Four hours of guessing that ONE traced request would have ended in minutes.

Every metric that team had was a *systems* metric - up/down, fast/slow, 200/500. None of them could answer the only question that mattered: **what did the AI actually send, get back, and how good was it?** Observability for LLMs adds that missing layer - a record of every call's input, output, tokens, cost, latency, AND quality - so "the answers got worse" becomes a query, not a four-hour guessing game. This lesson builds that layer with Langfuse, scores quality automatically, rolls it into dashboards, and ends with an app where every call is visible.

## Flying Blind vs Flying With Instruments

> ✈️ **Analogy**
>
> **Imagine flying a plane without instruments.** No speedometer, no altimeter, no fuel gauge. You're up there hoping everything is fine. Now add instruments: you see speed, altitude, fuel, engine temperature, wind direction in real-time. If anything goes wrong, you know INSTANTLY - and exactly what went wrong.
> Without observability, your AI app is that instrument-less plane. With Langfuse or LangSmith, every LLM call shows: what was sent, what came back, how long it took, how much it cost, and whether the output was good. **You see everything. You fix problems before users notice.**

> 💡 **Info**
>
> What you can't answer without observability
> - "Why is the bot giving wrong answers to billing questions?" (no trace of what prompt was sent)
> - "Why did our AI bill double last Tuesday?" (no cost breakdown per endpoint)
> - "Why are users complaining about slow responses?" (no latency metrics)
> - "Is our new prompt better than the old one?" (no quality comparison data)

> ℹ️ **Info**
>
> What you'll build
> - **Langfuse tracing** - instrument every LLM call with traces, spans, and metadata
> - **LangSmith tracing** - the LangChain ecosystem's observability tool
> - **Quality scoring** - score outputs automatically and attach scores to traces
> - **Dashboards** - cost tracking, latency monitoring, error rates
> - **Project** - fully instrumented chat app with end-to-end observability

> 📦 **Info**
>
> ⚠️ Misconception check: "My cloud provider's metrics already cover this"
> They cover the wrong layer. CPU, memory, latency, error rate, uptime - all green in the story above, and all useless for a QUALITY regression. LLM observability is a different axis: it records the *content* of each call (prompt, response) and a *judgment* of it (was it grounded? did the user like it?). Two traps that follow: (1) "no errors = healthy" - a model can return a confident, wrong answer with a perfect 200 status; only a quality SCORE catches that - trusting a green 200 is the classic anti-pattern here, exactly what NOT to do. (2) "logging is observability" - a flat wall of log lines cannot show you that ONE slow step inside ONE request; you need the nested trace tree (step 1), not grep.

## 60-Second Warm-Up: What You Already Know

Three recalls from earlier lessons - each is load-bearing today. Click a card to check yourself.

## Instrument It: Traces, Scores, and a Dashboard

The next four steps build observability by hand - trace an LLM call, score whether the answer was good, and roll traces into the five metrics every team watches. This is the transferable core; the tool and eval survey that follows (steps 5-9) maps the ecosystem, and step 10 assembles everything into one instrumented app.

---

## 🎯 Concept 1: Langfuse - Open-Source LLM Observability

### Langfuse - Open-Source LLM Observability

Free, self-hostable, works with any LLM. The most popular open-source option.

Start with the primitive everything else builds on: the trace. One user interaction becomes a TRACE; each step inside it (embed, retrieve, call the LLM) is a SPAN; the LLM call specifically is a GENERATION that carries tokens and cost; and a SCORE clips on to say whether it was any good. The v3 SDK is OpenTelemetry-based, so you open these with plain `with` blocks. Get this four-word vocabulary - trace, span, generation, score - and the rest of the lesson is detail.

You wrap one chat call in Langfuse. What does the dashboard let you do that a print() log never could?

> 🔍 **Analogy**
>
> **Think of Langfuse as CCTV for your AI.** Every LLM call is recorded: what went in (prompt), what came out (response), how long it took, what it cost, and whether the user liked it. You can rewind to any call, inspect every detail, and find patterns in failures. All without changing how your AI works - you just wrap your existing code.
> **Where this breaks down:** CCTV records passively; observability lets you REACT - a low score can page an engineer or flip a prompt back to the last good version. And unlike a camera, you control what is recorded: prompts with personal data can be redacted before storage (step 9), which matters a great deal in India.

**📝 `part1a_langfuse_setup.py`** - *language-python*

In [ ]:
# =============================================
# LANGFUSE: instrument your first LLM call (v3 SDK)
# pip install "langfuse>=3" google-genai
# Sign up free: https://cloud.langfuse.com  (ClickHouse acquired
# Langfuse in Jan 2026; it stays open-source and self-hostable.)
# =============================================

from langfuse import get_client
from google import genai
import os, time

# Reads LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY / LANGFUSE_HOST from env.
langfuse = get_client()
client = genai.Client()          # reads GOOGLE_API_KEY
MODEL = "gemini-2.5-flash"       # gemini-2.0-flash was shut down 2026-06-01

def ask_with_tracing(question: str, user_id: str = "anonymous") -> str:
    """Make an LLM call wrapped in a Langfuse generation (v3 context managers)."""
    # A generation = an LLM call. The context manager opens/closes the span
    # and makes it the CURRENT observation for anything nested inside.
    with langfuse.start_as_current_generation(
        name="gemini-call",
        model=MODEL,
        input=question,
        model_parameters={"temperature": 0.3},
    ) as generation:
        start = time.time()
        resp = client.models.generate_content(model=MODEL, contents=question)
        answer = resp.text

        # Attach the real numbers Langfuse costs and charts on:
        generation.update(
            output=answer,
            usage_details={
                "input_tokens": resp.usage_metadata.prompt_token_count,
                "output_tokens": resp.usage_metadata.candidates_token_count,
            },
            metadata={"latency_ms": round((time.time() - start) * 1000)},
        )
        # Trace-level metadata (who/where) belongs on the enclosing trace:
        langfuse.update_current_trace(user_id=user_id, metadata={"source": "web"})
    return answer

answer = ask_with_tracing("What is RAG in 2 sentences?", user_id="student_001")
print(f"Answer: {answer}")
langfuse.flush()   # short-lived script: force-send before exit

# Output:
#   Answer: RAG (Retrieval-Augmented Generation) retrieves relevant documents
#   first, then feeds them to the LLM so answers are grounded in real data...
#   -> a "gemini-call" generation now appears in your Langfuse dashboard with
#      input, output, 41 input / 58 output tokens, cost, and latency.


**📝 `part1b_langfuse_decorator.py`** - *language-python*

In [ ]:
# =============================================
# LANGFUSE DECORATOR: zero-boilerplate instrumentation (v3)
# Just add @observe() to any function - it becomes a span.
# =============================================

from langfuse import observe, get_client

langfuse = get_client()

@observe()                       # a plain span
def process_query(question: str) -> str:
    """Full pipeline: embed -> retrieve -> generate. Each step auto-traced."""
    embedding = embed_question(question)
    docs = retrieve_documents(embedding)
    answer = generate_response(question, docs)
    langfuse.update_current_span(metadata={"num_docs": len(docs), "pipeline": "rag_v2"})
    return answer

@observe()
def embed_question(text: str) -> list:
    r = client.models.embed_content(model="gemini-embedding-001", contents=text)
    return r.embeddings[0].values

@observe()
def retrieve_documents(embedding: list) -> list[str]:
    # In production: query your vector database here.
    return ["RAG combines retrieval with generation...",
            "Vector databases store embeddings..."]

@observe(as_type="generation")   # a GENERATION: tracks model, tokens, cost
def generate_response(question: str, docs: list[str]) -> str:
    prompt = f"Context: {chr(10).join(docs)}\n\nQuestion: {question}\nAnswer:"
    resp = client.models.generate_content(model=MODEL, contents=prompt)
    langfuse.update_current_generation(
        model=MODEL, input=prompt, output=resp.text,
        usage_details={"input_tokens": resp.usage_metadata.prompt_token_count,
                       "output_tokens": resp.usage_metadata.candidates_token_count})
    return resp.text

result = process_query("How does RAG work?")
print(f"Answer: {result[:100]}...")

# Output: (what the Langfuse dashboard now shows - a nested tree)
#   Trace: process_query
#     |- Span: embed_question         (latency 120ms)
#     |- Span: retrieve_documents     (latency  45ms)
#     |- Generation: generate_response
#          model gemini-2.5-flash | in 156 tok | out 89 tok | 1,230ms | $0.00027


#### 💡 What just happened

What just happened?
  Two instrumentation approaches: (1) **Manual tracing** - create trace → create generation → end with output/usage. Full control over what's tracked. (2) **Decorator-based** - just add `@observe()` to any function. Langfuse automatically creates spans for each function call and nests them. The `as_type="generation"` decorator specifically tracks LLM calls with tokens and cost. Both approaches show the same result in the dashboard: a timeline of every step with latency, tokens, and cost.

- Scene 1: a wall of flat log lines - impossible to see which step was slow or costly.

- Scene 2: the same request becomes ONE trace bar (one user interaction).

- Scene 3: the tree grows - spans (embed, retrieve) nest inside, and a generation carries model + tokens + cost + latency.

- Scene 4: a score chip (thumbs, LLM-judge 0.85, groundedness 0.92) clips on - trace + span + generation + score is the whole model.

Controls: Play/Pause, Reset, speed (0.5x/1x/2x). Scene buttons jump; the caption narrates.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: LangSmith - The LangChain Ecosystem Tool

### LangSmith - The LangChain Ecosystem Tool

Built by the LangChain team. Deep integration with LangChain and LangGraph.

Same four concepts, a second vendor. LangSmith is Langfuse's closest peer, built by the LangChain team; if your stack already lives in LangChain/LangGraph, its `@traceable` decorator drops in with almost no thought. Seeing it right after Langfuse makes the point that these are two spellings of one idea - pick by your stack, not by fear of missing out.

LangSmith and Langfuse are different products. How different are the core CONCEPTS?

> 🔖 **Analogy**
>
> **Two dashcam brands, one road.** Langfuse and LangSmith are like two dashcam makers: different apps, different mounts, but both record the same drive - who you called, what was said, how long, what it cost. `@traceable` is LangSmith's mount; `@observe` is Langfuse's. Learn to read one recording and you can read the other.
> **Where this breaks down:** the brands are not interchangeable at switch-time - traces live in whichever backend you chose, and migrating history is real work. The vendor-neutral escape hatch is OpenTelemetry (step 5): instrument once against the OTEL standard and export to either.

**📝 `part2_langsmith.py`** - *language-python*

In [ ]:
# =============================================
# LANGSMITH: tracing with the LangChain ecosystem
# pip install langsmith
# Sign up: https://smith.langchain.com
# =============================================

import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY", "")
os.environ["LANGSMITH_PROJECT"] = "netsetos-genai-course"

from langsmith import traceable
from google import genai

client = genai.Client()
MODEL = "gemini-2.5-flash"

@traceable(name="rag_pipeline")
def rag_pipeline(question: str) -> str:
    """Full RAG pipeline - every @traceable step nests automatically."""
    docs = retrieve(question)
    return generate(question, docs)

@traceable(name="retrieve")
def retrieve(query: str) -> list[str]:
    return ["RAG retrieves relevant context before generating..."]

@traceable(name="generate", run_type="llm")   # run_type="llm" -> LangSmith
def generate(question: str, docs: list[str]) -> str:   # tracks it as an LLM span
    context = "\n".join(docs)
    resp = client.models.generate_content(
        model=MODEL, contents=f"Context: {context}\n\nQ: {question}\nA:")
    return resp.text

answer = rag_pipeline("What is vector search?")
print(f"Answer: {answer[:100]}...")

# Output:
#   Answer: Vector search finds items by semantic similarity, comparing
#   embedding vectors with a distance metric like cosine similarity...
#   -> a "rag_pipeline" trace with nested retrieve + generate runs in LangSmith.
# Same four concepts as Langfuse (trace, span, generation, score) - different UI.


> ✅ **Info**
>
> Langfuse vs LangSmith: which to choose?
> - **Langfuse** - open-source, self-hostable, works with any framework, free tier generous. Best for: teams that want control and vendor-independence.
> - **LangSmith** - built by LangChain team, deep LangChain/LangGraph integration, better evaluation tools. Best for: teams already using LangChain.
> - **Both** are excellent. Pick based on your stack. The concepts (traces, spans, generations, scores) are identical.

---

## 🎯 Concept 3: Quality Scoring - Was the Answer Actually Good?

### Quality Scoring - Was the Answer Actually Good?

Latency and cost don't tell you if the answer was helpful. Scoring does.

This is the layer the cold-open team was missing. A trace tells you what happened; a SCORE tells you whether it was any good - the axis that stays invisible to latency and error dashboards. Three sources, cheapest to richest: the user's thumb, a cheap judge model (lesson 7.5's pattern, now attached to the trace), and a groundedness check that catches made-up facts.

Latency and cost are both green. Can you conclude the answer was good?

> ⭐ **Analogy**
>
> **Think of a restaurant review system.** The kitchen tracks how fast they cook (latency) and how much ingredients cost (cost). But neither tells you if the food was *good*. For that, you need customer reviews (user feedback) and food critics (LLM judge scores). Quality scoring adds the "was it good?" dimension to your observability.

**📝 `part3_quality_scoring.py`** - *language-python*

In [ ]:
# =============================================
# QUALITY SCORING: 3 ways to score outputs (v3 SDK)
#   1. User feedback (thumbs)  2. LLM-as-judge  3. Groundedness
# Scores attach to a trace so you can chart quality over time.
# =============================================

from langfuse import get_client
from google import genai

langfuse = get_client()
client = genai.Client()
JUDGE = "gemini-2.5-flash"        # a cheap, separate judge model

class QualityScorer:
    """Score LLM outputs and attach the scores to a Langfuse trace."""

    def score_user_feedback(self, trace_id: str, thumbs_up: bool):
        """Record whether the user liked the response (a numeric score)."""
        langfuse.create_score(
            trace_id=trace_id, name="user-feedback",
            value=1 if thumbs_up else 0,
            comment="thumbs up" if thumbs_up else "thumbs down")

    def score_llm_judge(self, trace_id: str, question: str, answer: str) -> float:
        """Use a cheap LLM to rate answer quality 0.0-1.0."""
        prompt = (f"Rate this answer's quality from 0.0 to 1.0.\n"
                  f"Question: {question[:200]}\nAnswer: {answer[:500]}\n"
                  f"Criteria: accuracy, completeness, clarity. Reply with ONLY a number.")
        resp = client.models.generate_content(
            model=JUDGE, contents=prompt,
            config={"temperature": 0.0})     # deterministic judging
        try:
            score = max(0.0, min(1.0, float(resp.text.strip())))
        except ValueError:
            score = 0.5                       # judge did not return a clean number
        langfuse.create_score(trace_id=trace_id, name="llm-judge", value=score,
                              comment=f"auto-scored by {JUDGE}")
        return score

    def score_groundedness(self, trace_id: str, answer: str, context: str) -> float:
        """Is the answer supported by the context? (catches hallucinations)"""
        prompt = (f"Is this answer FULLY supported by the context? Score 0.0-1.0.\n"
                  f"0.0 = makes up facts not in context; 1.0 = every claim is in context.\n"
                  f"Context: {context[:500]}\nAnswer: {answer[:500]}\nScore:")
        resp = client.models.generate_content(model=JUDGE, contents=prompt)
        try:
            score = max(0.0, min(1.0, float(resp.text.strip())))
        except ValueError:
            score = 0.5
        langfuse.create_score(trace_id=trace_id, name="groundedness", value=score)
        return score

# -- Demo: trace one call, then score THAT trace by id --
scorer = QualityScorer()
question = "What is the difference between RAG and fine-tuning?"

with langfuse.start_as_current_generation(name="scored-chat", model="gemini-2.5-flash",
                                          input=question) as gen:
    resp = client.models.generate_content(model="gemini-2.5-flash", contents=question)
    answer = resp.text
    gen.update(output=answer)
    trace_id = langfuse.get_current_trace_id()   # the id these scores attach to

scorer.score_user_feedback(trace_id, thumbs_up=True)
quality = scorer.score_llm_judge(trace_id, question, answer)
print(f"LLM judge score: {quality:.2f}")
langfuse.flush()

# Output:
#   LLM judge score: 0.88
#   In the dashboard this ONE trace now carries three scores:
#     user-feedback = 1 | llm-judge = 0.88 | groundedness = 0.92
#   Over thousands of traces you chart avg quality, compare prompts, and
#   jump straight to the traces where quality dropped.


#### 💡 What just happened

What just happened?
  Three scoring methods attached to traces: (1) **User feedback** - simple thumbs up/down from the UI. (2) **LLM judge** - automatic quality score (0-1) using a cheap model as evaluator. (3) **Groundedness check** - verifies the answer is supported by the context (catches hallucinations). All scores appear on the trace in the dashboard, enabling you to track quality trends over time, compare prompts, and investigate drops.

---

## 🎯 Concept 4: Dashboards - The Metrics That Matter

### Dashboards - The Metrics That Matter

Turn raw traces into actionable insights: cost trends, latency percentiles, quality over time.

Thousands of individual traces are noise until you aggregate them. This step computes the five numbers a team actually watches - and one word worth pinning down now: a percentile. P95 latency is the value that ninety-five percent of requests come in UNDER; it is the honest headline number, because an average hides the slow tail that users actually feel.

Your average latency is about 800ms and looks fine. Why might users still be furious?

> 🛀 **Analogy**
>
> **A car's dashboard, not its logbook.** The logbook (raw traces) records every trip in detail - invaluable when something breaks. But you do not drive by reading the logbook; you drive by the DASHBOARD: speed, fuel, engine temp, a warning light. Your five gauges are the same - volume, latency, cost, quality, per-model - a glanceable summary that turns a million rows into "are we healthy right now?".
> **Where this breaks down:** a fuel gauge at a fixed threshold ("alert near empty") is fine for a tank; production alerting on a raw threshold ("alert if latency > 2s") floods you with false alarms during normal spikes. The fix is burn-rate alerting (step 8): alert when a metric is degrading FAST relative to its own baseline, not when it crosses a static line.

**📝 `part4_metrics_exporter.py`** - *language-python*

In [ ]:
# =============================================
# METRICS EXPORTER: turn raw traces into the 5 numbers
# every production AI team watches (dashboards + alerts).
# (Langfuse/LangSmith ship these dashboards; we compute them
#  by hand once so the numbers stop being magic.)
# =============================================

from collections import defaultdict
from datetime import datetime, timedelta
import statistics

def percentile(values, p):
    """P95 = 'only 5% of requests are slower than this'. Nearest-rank."""
    if not values:
        return 0
    ordered = sorted(values)
    k = max(0, min(len(ordered) - 1, round(p / 100 * len(ordered)) - 1))
    return ordered[k]

class ObservabilityDashboard:
    """Track and report key LLM metrics from recorded traces."""

    def __init__(self):
        self.traces = []

    def record(self, trace_data: dict):
        trace_data["timestamp"] = datetime.now()
        self.traces.append(trace_data)

    def report(self, hours: int = 24):
        cutoff = datetime.now() - timedelta(hours=hours)
        recent = [t for t in self.traces if t["timestamp"] >= cutoff]
        if not recent:
            print(f"No traces in the last {hours} hours.")
            return

        print(f"\nObservability Dashboard (last {hours}h)")
        print("=" * 52)

        # 1) Volume + error rate
        errors = [t for t in recent if t.get("error")]
        print(f"  Requests: {len(recent)}   Error rate: {len(errors)/len(recent):.1%}")

        # 2) Latency percentiles
        lat = [t["latency_ms"] for t in recent if "latency_ms" in t]
        if lat:
            print(f"  Latency  P50 {percentile(lat,50):>5.0f}ms  "
                  f"P95 {percentile(lat,95):>5.0f}ms  P99 {percentile(lat,99):>5.0f}ms")

        # 3) Cost (total, per-call, projected monthly)
        costs = [t["cost_usd"] for t in recent if "cost_usd" in t]
        if costs:
            print(f"  Cost     total ${sum(costs):.4f}  avg ${statistics.mean(costs):.5f}"
                  f"  projected/mo ${sum(costs)/hours*720:.2f}")

        # 4) Quality
        scores = [t["quality_score"] for t in recent if "quality_score" in t]
        if scores:
            low = [x for x in scores if x < 0.6]
            print(f"  Quality  avg {statistics.mean(scores):.2f}  "
                  f"low(<0.6) {len(low)} ({len(low)/len(scores):.0%})")

        # 5) Per-model split
        by_model = defaultdict(lambda: {"n": 0, "cost": 0.0})
        for t in recent:
            m = by_model[t.get("model", "unknown")]
            m["n"] += 1
            m["cost"] += t.get("cost_usd", 0)
        print("  By model:")
        for model, d in by_model.items():
            print(f"    {model:22s} {d['n']:>4} calls  ${d['cost']:>7.4f}")

# -- Simulate 20 traced calls --
import random
dash = ObservabilityDashboard()
for _ in range(20):
    dash.record({
        "model": random.choice(["gemini-2.5-flash", "gpt-5.4-mini"]),
        "latency_ms": random.randint(200, 3000),
        "cost_usd": random.uniform(0.0001, 0.005),
        "quality_score": random.uniform(0.5, 1.0),
        "error": random.random() < 0.05,
    })
dash.report(hours=24)

# Output:
#   Observability Dashboard (last 24h)
#   Requests: 20   Error rate: 5.0%
#   Latency  P50  1620ms  P95  2890ms  P99  2980ms
#   Cost     total $0.0483  avg $0.00242  projected/mo $1.45
#   Quality  avg 0.74  low(<0.6) 3 (15%)
#   By model:
#     gemini-2.5-flash        11 calls  $0.0261
#     gpt-5.4-mini             9 calls  $0.0222


#### 💡 What just happened

What just happened?
  We built an `ObservabilityDashboard` that computes the 5 key metrics from traces: (1) Volume + error rate, (2) Latency percentiles (P50, P95, P99), (3) Cost (total, per-call average, projected monthly), (4) Quality (average score, low-quality percentage), (5) Per-model breakdown. These are the exact metrics every production AI team monitors. In real production, Langfuse and LangSmith provide pre-built dashboards with these metrics - our code replicates the analytics for understanding.

- Scene 1: a blank instrument panel - a latency and cost spike happen invisibly.

- Scene 2: traces stream into a collector; volume ticks up and a few flash red (errors).

- Scene 3: five gauges fill - volume+errors, latency (P50/P95/P99), cost, quality, per-model. P95 is the value ninety-five percent of requests come in under.

- Scene 4: the quality gauge drifts below 0.5 and a burn-rate alarm fires while latency and cost stay green.

Controls: Play/Pause, Reset, speed. Scenes 3-4 are the ObservabilityDashboard above, animated.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## The Ecosystem: Tools, Evals, and Production Reality

Steps 1-4 are the engine. Steps 5-9 are the field guide: the tool landscape, Langfuse's deeper features, the evaluation frameworks that answer "is it actually good?", production monitoring, and what India's DPDP Act demands. Then step 10 wires it all into one app.

---

## 🎯 Concept 5: Tool Landscape - Helicone, Phoenix, Portkey, OpenTelemetry

### Tool Landscape - Helicone, Phoenix, Portkey, OpenTelemetry

Five tools, five architectures. When to use which.

Langfuse and LangSmith are not the only options - and the differences are mostly ARCHITECTURAL, which tells you when to reach for each. Do not memorize the table; learn the two axes it varies on: how the tool taps into your calls (a proxy in the request path vs a decorator in your code vs a vendor-neutral OTEL exporter) and what it optimizes for (instant cost visibility vs deep tracing vs evaluation vs governance).

Helicone works by changing one base URL; Langfuse works by decorators. What does the URL trick cost you?

> 📡 **Analogy**
>
> **How you tap a phone line.** A PROXY tool (Helicone) splices into the line itself - one URL change, zero code, but now it sits in your call path (if it hiccups, so can your calls). A DECORATOR tool (Langfuse, W&B Weave) taps from inside your own code - more to write, full control, no extra hop. An OTEL exporter (OpenLLMetry) is a standard jack any recorder can plug into - instrument once, fan out to Grafana, Langfuse, and Datadog at the same time. Same signal, three tap points, three trade-offs.

| Tool | Architecture | Free Tier | Self-Host | Best For |
|---|---|---|---|---|
| Helicone | API proxy (1-line) | free tier | ✅ open-source | Fastest setup, cost tracking |
| Arize Phoenix | OTEL-native | 25K spans/mo | ✅ ELv2 | OSS tracing, notebook-first |
| Portkey | AI gateway | 10K logs/mo | ✅ Gateway OSS | Enterprise governance, guardrails |
| Braintrust | Eval-first | 1GB data | ❌ Cloud only | Systematic evaluation, CI/CD |
| W&B Weave | @weave.op decorator | 1GB/mo | ✅ Enterprise | Full ML lifecycle teams |

> ✅ **Info**
>
> 🌐 OpenTelemetry GenAI Semantic Conventions
> **OpenLLMetry** (Traceloop, open-source, Apache 2.0): one line `Traceloop.init()` auto-instruments 20+ providers. OTEL semantic conventions define `gen_ai.request.model`, `gen_ai.usage.input_tokens`, `gen_ai.operation.name`. Export to any backend: Grafana, Langfuse, Datadog, Jaeger simultaneously. Prompt/completion content NOT captured by default (privacy). Datadog and other backends adopted these conventions. **OTEL is the vendor-neutral, future-proof tap point.**

**📝 `taps.py`** - *language-python*

```python
# The three tap points, in code:

# 1) PROXY (Helicone) - change one base_url, zero code changes:
from openai import OpenAI
client = OpenAI(base_url="https://oai.helicone.ai/v1",
                default_headers={"Helicone-Auth": "Bearer <key>"})

# 2) DECORATOR (Langfuse) - from step 1:
from langfuse import observe
@observe()
def my_call(q): ...

# 3) OTEL EXPORTER (OpenLLMetry) - one init, fan out to many backends:
from traceloop.sdk import Traceloop
Traceloop.init(app_name="netsetos")   # -> Grafana + Langfuse + Datadog at once

# Output: same signal, three taps. The proxy is in your call path; the
# decorator lives in your code; the OTEL exporter is vendor-neutral.
```

#### 💡 What just happened

What just happened?**Decision framework (by the tap point):** zero-code proxy → Helicone (note: acquired by Mintlify in Mar 2026, now largely in maintenance mode - fine to use, but weigh its roadmap). Free OSS, OTEL-native → Phoenix. Enterprise AI gateway → Portkey. Eval-first → Braintrust. Already on W&B → Weave. Vendor-neutral → OpenTelemetry + OpenLLMetry. **The recommended stack:** Langfuse (deep tracing + prompt management) + Helicone (instant cost visibility) + OpenLLMetry (vendor-neutral OTEL) + Grafana (dashboards + alerting).

---

## 🎯 Concept 6: Advanced Langfuse - Prompts, Datasets, Experiments, Annotation Queues

### Advanced Langfuse - Prompts, Datasets, Experiments, Annotation Queues

Prompt versioning. Experiment runner. Human review. Self-hosting v3.

Tracing is the entry point; the reason teams standardize on Langfuse is the platform ON TOP of it. Prompt management lets product edit prompts without a deploy; datasets + experiments turn "I think the new prompt is better" into a measured comparison; annotation queues bring human experts into the loop. And the v3 architecture (ClickHouse) is what let it scale - the same ClickHouse whose parent company acquired Langfuse in January 2026.

Why do teams outgrow "just tracing" and want prompt management + datasets + experiments?

> 📁 **Analogy**
>
> **From a security camera to a whole studio.** Step 1's Langfuse was a camera - it recorded. This is the editing studio around it: a versioned script library (prompt management - swap the script without rebuilding the set), a screening room to compare two cuts against the same test audience (datasets + experiments), and a review desk where human editors mark up takes (annotation queues). Recording was step one; producing is the job.

> 📦 **Info**
>
> 🚀 Langfuse v3 architecture - and the ClickHouse acquisition
> - **Why v3:** at scale PostgreSQL was the bottleneck (slow prompt API, dashboard queries timing out), so v3 moved analytics onto the [ClickHouse](https://clickhouse.com/blog/clickhouse-acquires-langfuse-open-source-llm-observability) columnar database. Two years later (Jan 2026) ClickHouse the COMPANY acquired Langfuse - it stays MIT-licensed and self-hostable, same pricing.
> - **Result:** the prompt API and dashboard queries went from seconds to sub-second, and thousands of teams now self-host it.
> - **Architecture:** 6 services - Web (UI+API), Worker (async processing), ClickHouse (analytics), Redis (queue+cache), S3/MinIO (event storage), PostgreSQL (accounts, prompts, configs).
> - **Self-host:** `git clone langfuse && docker compose up` - ready at localhost:3000 in ~3 minutes. Production: 2 CPUs, 4GB RAM per container minimum.

> ✅ **Info**
>
> 📊 Datasets, Experiments, Annotation Queues
> - **Datasets:** Versioned (timestamp snapshots), folders (`/` in names), batch-add from production observations. Start with a few dozen examples.
> - **Experiment Runner:** Define task function + evaluators → run against dataset → auto-trace + side-by-side comparison. Runs concurrently, isolates failures.
> - **Annotation Queues:** Domain experts review and score traces. Score configs (e.g., "Relevance" 1-5). Session-level annotations for multi-turn review. Full CRUD API.
> - **SDK (v3/v4):** OpenTelemetry-native (the SDK you used in steps 1-3). `@observe()` decorator, context managers for spans/generations, and it accepts traces from any OTEL-instrumented app via the OTLP endpoint.

**📝 `prompt_mgmt.py`** - *language-python*

In [ ]:
# Prompt management: edit the prompt in the Langfuse UI, fetch it in code -
# no deploy needed to change a prompt.
from langfuse import get_client
langfuse = get_client()

# Fetch the version currently labelled "production" (cached client-side):
prompt = langfuse.get_prompt("support-bot", label="production")
compiled = prompt.compile(question="How do I reset my password?")   # fills {{question}}

resp = client.models.generate_content(model="gemini-2.5-flash", contents=compiled)

# Output: product edits the prompt in the UI and relabels a new version
# "production"; this code picks it up on the next fetch. Rollback = relabel
# the old version. Prompt iteration is decoupled from code deployment.


#### 💡 What just happened

What just happened? Langfuse v3 is a **production-grade platform**, not just a tracing SDK. Prompt management with A/B testing separates prompt iteration from code deployments. Datasets + Experiments enable systematic offline evaluation. Annotation queues bring human review into the loop. The ClickHouse migration solved the scaling bottleneck that plagued v2. **Pricing:** a free hobby tier, then paid Core and Pro tiers; self-hosted open-source is free with all features (verify current numbers on langfuse.com/pricing).

---

## 🎯 Concept 7: Evaluation Frameworks - RAGAS, DeepEval, LLM-as-Judge

### Evaluation Frameworks - RAGAS, DeepEval, LLM-as-Judge

Faithfulness, relevancy, context metrics. Pytest for LLMs. Judge patterns.

Step 3 scored answers by hand; production wants batteries-included metrics. Two frameworks own this space: RAGAS speaks RAG (faithfulness = does every claim trace back to the retrieved docs?), and DeepEval speaks pytest (turn quality checks into CI gates that fail the build when scores drop). Underneath both is the LLM-as-judge pattern from 7.5, with a few rules that make it trustworthy.

You want to know WHICH thing is wrong with a RAG answer, not just that something is. What gives you that?

> 🔬 **Analogy**
>
> **A blood panel, not a bathroom scale.** "The output looked fine" is a bathroom scale - one crude number. RAGAS is a blood panel: it draws the answer apart into named markers (faithfulness, relevancy, context precision, context recall) so you know WHICH thing is wrong, not just that something is. Faithfulness is the hallucination marker: it extracts each claim and checks it against the retrieved context, one by one.
> **Where this breaks down:** a blood panel has calibrated instruments; an LLM judge is itself a fallible model. So you calibrate it - ask for a rationale before the score, prefer binary verdicts over fuzzy 1-5 scales, keep the judge separate from the generator, and spot-check against human labels. A judge you never audit is a scale you never zeroed.

| Framework | Metrics | Key Feature | Integration |
|---|---|---|---|
| RAGAS | Faithfulness, Relevancy, Context Precision/Recall, and more | Reference-free RAG eval | Langfuse, LangChain |
| DeepEval | many (RAG, agents, safety, chat) | pytest integration, CI/CD | Native pytest + Confident AI |
| LLM-as-Judge | Custom rubrics | high human agreement (well-designed) | Any (custom code) |

> 💡 **Info**
>
> ⚠️ LLM-as-Judge best practices
> **Ask for rationale BEFORE the score** (not after). Use **binary yes/no** over multi-point scales. Separate judge from generator model. Randomize option order in pairwise comparisons (position bias). A majority-vote ensemble increases reliability. Use a stronger model for high-stakes checks and a cheap one for volume. Well-designed judges reach near human-level agreement in published studies; a bare-number judge, or a judge that is the same model as the generator, does not.

#### 💡 What just happened

What just happened?**The evaluation flywheel** is the pattern to remember (step 3 scoring, industrialized): online monitoring catches issues → add the failure to a golden dataset → offline experiments stop it regressing → repeat. RAGAS for RAG (faithfulness = hallucination detection). DeepEval for pytest-style CI/CD quality gates (fail builds when quality drops). LLM-as-judge for custom quality metrics. **Online eval:** score a small sample of production traffic (a low single-digit percentage) automatically. **Offline eval:** run against golden datasets before deployment. Both are necessary.

- Scene 1: LLM-as-judge the right way - rationale FIRST, then a verdict; vs a noisy "just give a number".

- Scene 2: online monitoring samples ~5% of live traffic; one trace scores low on faithfulness (a hallucinated claim highlighted).

- Scene 3: the failing case is added to a growing golden dataset (every real failure becomes a permanent test).

- Scene 4: an offline experiment runs the dataset against a new prompt version, v2 beats v1, the fix ships, and the loop turns back to monitoring.

Controls: Play/Pause, Reset, speed. The loop is the whole point - replay it end to end.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 8: Production Monitoring - OpenTelemetry, Grafana, Drift Detection

### Production Monitoring - OpenTelemetry, Grafana, Drift Detection

OTEL instrumentation. SLO-based alerting. Model drift. Agent tracing.

Langfuse traces answer "what happened to this request?"; production monitoring answers "is the whole system healthy, right now, and who do I page if not?". This is where LLM observability rejoins classic SRE - OpenTelemetry for vendor-neutral pipes, Grafana for dashboards and alerts, and one LLM-specific hazard the cold open already showed you: silent model drift.

An alert that fires "if latency > 2s" keeps waking you during normal traffic spikes. Better design?

> 🚨 **Analogy**
>
> **Smoke detectors vs a fire inspector.** Raw-threshold alerts ("latency > 2s") are smoke detectors that shriek every time you make toast - so people rip out the batteries. Burn-rate alerting is a fire inspector: it fires when a metric is degrading FAST relative to its own baseline (error rate 2x normal, sustained 5 minutes), so an alert means something. And silent drift is the fire with no smoke - quality falls while latency and errors stay flat; only a daily golden-dataset eval smells it.
> **Where this breaks down:** a fire inspector needs a baseline of "normal" to judge against, and LLM baselines shift as your traffic changes - so you re-baseline periodically and keep a frozen golden dataset as the fixed reference point. Alert on the golden set, trend on live traffic.

> ✅ **Info**
>
> 📈 Production monitoring stack
> - **Instrumentation:** OpenLLMetry `Traceloop.init()` → OTEL Collector → fan out to Grafana + Langfuse + Datadog at once ([OTEL GenAI semantic conventions](https://github.com/open-telemetry/semantic-conventions/tree/main/docs/gen-ai)). Overhead is negligible per call.
> - **Key metrics:** P50/P95/P99 latency, TTFB, tokens per request, cost per request/user/feature, error rates, rate limit hits, quality scores (faithfulness, relevancy).
> - **Alerting (SLO burn rates, not raw thresholds):** a burn rate is how fast you are eating your error budget - alert when error rate runs several times normal for a sustained window, when P95 latency stays well above baseline, when hourly spend spikes far over the weekly average, or when faithfulness drops sharply from its baseline.
> - **Model drift:** Embedding-based detection - capture prompt/response embeddings, compare distributions over time. Alert → LLM-as-judge classifies drift type (topic shift, quality change, style change).
> - **Agent tracing:** Langfuse trace trees with typed observations (spans for planning/tools, generations for LLM calls). Cross-service propagation via trace_id/span_id.

#### 💡 What just happened

What just happened? Production monitoring goes beyond Langfuse traces. **OpenTelemetry** provides vendor-neutral instrumentation exportable to any backend. **Grafana** provides dashboards + alerting with SLO burn rates (not raw thresholds - those create noise). **Drift detection** catches silent provider model updates that degrade quality. **Agent tracing** requires specialized patterns - nested trace trees, cross-service propagation, streaming observability (TTFB, tokens/second).

- Scene 1: an ops panel - latency, error rate, throughput - all green and steady.

- Scene 2: user complaints pile up ("answers got worse") while the ops gauges stay stubbornly green.

- Scene 3: the cause - the provider silently updated the model behind the same name; the hidden quality gauge has quietly dropped.

- Scene 4: a daily golden-dataset eval fires a QUALITY alert and classifies the drift - the only thing that catches it. Pin versions, eval daily.

Controls: Play/Pause, Reset, speed. This is the four-hour outage from the cold open, in 30 seconds.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 9: India Compliance - DPDP, Self-Hosting, INR Cost Tracking

### India Compliance - DPDP, Self-Hosting, INR Cost Tracking

Data residency. PII redaction. ₹ dashboards. Sarvam AI evaluation.

Observability means logging what your AI saw and said - which, for Indian users, is often personal data under the DPDP Act. That turns a nice-to-have into a design constraint: WHERE the traces live, WHAT is redacted before they land, and how costs read in rupees. This step is the compliance-shaped version of everything above.

Your users are in India and your bot logs their prompts. What does DPDP force you to change first?

> 🏛️ **Analogy**
>
> **A bank vault with a redaction desk at the door.** You would not keep customer records in a warehouse in another country - DPDP says the vault must be on Indian soil (self-host in Mumbai; Langfuse Cloud has no India region). And before any document enters the vault, a clerk blacks out the Aadhaar and phone numbers (PII redaction BEFORE trace ingestion). Inside, the ledger is kept in rupees (INR dashboards with GST), because that is the currency your finance team actually reconciles.
> **Where this breaks down:** a vault protects what is inside it, but DPDP also grants the right to ERASURE - a customer can demand their data be deleted, so you need to find and remove one user's traces on request. Redaction reduces what you hold; erasure capability handles what you still do.

> 📦 **Info**
>
> 🇮🇳 DPDP Act compliance for LLM observability
> - **Timeline:** DPDP Rules notified Nov 2025; consent-manager provisions land Nov 2026 and full enforcement is May 13, 2027 (an 18-month clock). Penalties run up to ~₹250 Cr for serious breaches.
> - **What you CAN log:** Anonymized data, aggregated metrics (latency, tokens, cost), system metadata, prompt templates, model config.
> - **What needs care:** User prompts with PII (names, Aadhaar, phone, health). Model outputs reflecting personal data. Session IDs linkable to users.
> - **PII redaction:** Protecto.ai (India-focused, multilingual PII detection including Indic languages) or Microsoft Presidio with Indic NER. Redact BEFORE trace ingestion.
> - **Data residency:** Langfuse Cloud has NO India region - self-host on AWS Mumbai (ap-south-1). Cost is roughly tens of thousands of rupees a month for dev/test, more for production (sizing-dependent).
> - **INR cost tracking:** Track in USD within tools, build Grafana layer converting to rupees using RBI reference rates plus the applicable GST on OIDAR services. Convert provider USD prices to INR at the current forex rate and add GST (OIDAR services) for the true landed cost.

> 💡 **Info**
>
> 📊 Evaluating Indic language models
> **Standard metrics (BLEU, ROUGE) are unreliable for Indic languages** - use Character-F1 (ChrF, a character-level overlap score that handles rich morphology better) instead. Indic safety benchmarks report large gaps in cross-language safety agreement - the same model can be much safer in English than in a low-resource Indian language. Key benchmarks: IndicGenBench (29 languages, 13 scripts), IndicSafe (6,000 culturally grounded safety prompts), PARIKSHA (combined human + LLM eval). **Log language as trace metadata** and track per-language quality metrics separately.

**📝 `pii_redaction.py`** - *language-python*

In [ ]:
# DPDP: redact PII BEFORE the trace is stored (never log raw Aadhaar/phone).
import re
def redact(text: str) -> str:
    text = re.sub(r"\d{4}\s?\d{4}\s?\d{4}", "[AADHAAR]", text)   # 12-digit Aadhaar
    text = re.sub(r"(?:\+91[- ]?)?[6-9]\d{9}", "[PHONE]", text)      # Indian mobile
    return text

from langfuse import observe
@observe()
def handle(user_msg: str) -> str:
    safe = redact(user_msg)          # what Langfuse stores is already masked
    return answer(safe)
# For whole functions you never want captured: @observe(capture_input=False).

# Output:
#   in : "my aadhaar is 1234 5678 9012, call me on 9876543210"
#   logged: "my aadhaar is [AADHAAR], call me on [PHONE]"
#   The trace is useful for debugging; the personal data never lands in it.


#### 💡 What just happened

What just happened? India's DPDP Act creates a **hard requirement for self-hosted observability** with PII redaction. Langfuse Cloud has no India region - self-host on AWS Mumbai. Track costs in INR including GST. For Indic language models (Sarvam AI), standard evaluation metrics fail - use ChrF and Indic-specific benchmarks. **The pattern:** self-host Langfuse in Mumbai + PII redaction with Protecto.ai + INR Grafana dashboards + per-language quality tracking.

---

## 🎯 Concept 10: Project: Fully Instrumented Chat App - Everything, Integrated

### Project: Fully Instrumented Chat App - Everything, Integrated

Combine everything: Langfuse tracing + quality scoring + metrics + alerts.

Rehearsals over. This wires step 1's tracing, step 3's scoring, and step 4's dashboard into one chat class where every message is traced, auto-scored, costed, and alertable - the exact capability whose absence cost the cold-open team four hours. Read it as the answer to that incident: with this running, "the answers got worse on Tuesday" is a dashboard filter, not a mystery.

With the fully instrumented app running, how does "answers got worse on Tuesday" get resolved?

> ✈️ **Analogy**
>
> **The full cockpit, finally.** The lesson opened with an instrument-less plane. This is the assembled panel: the flight recorder (Langfuse trace), the gauges (dashboard), the quality warning light (LLM-judge score), and the alarm (low-quality alert) - all fed by one instrumented `chat()` call. One class, every instrument live.

**📝 `project_instrumented_chat.py`** - *language-python*

In [ ]:
# =============================================
# FULLY INSTRUMENTED CHAT APP - everything, integrated
# Every call: traced (Langfuse v3), auto-scored, dashboarded, alerted.
# pip install "langfuse>=3" google-genai
# Reuses QualityScorer (step 3) and ObservabilityDashboard (step 4).
# =============================================

from langfuse import get_client
from google import genai
import time

langfuse = get_client()
client = genai.Client()
MODEL = "gemini-2.5-flash"

class InstrumentedChat:
    """Chat app with end-to-end observability."""

    def __init__(self):
        self.scorer = QualityScorer()
        self.dashboard = ObservabilityDashboard()
        self.system = "You are a helpful Netsetos tutor."

    def chat(self, message: str, user_id: str = "anon", session_id: str = "default") -> dict:
        # One generation per message; the context manager IS the trace boundary.
        with langfuse.start_as_current_generation(
                name="chat", model=MODEL, input=message) as gen:
            start = time.time()
            error = False
            try:
                resp = client.models.generate_content(
                    model=MODEL, contents=message,
                    config={"system_instruction": self.system})
                answer = resp.text
                in_tok = resp.usage_metadata.prompt_token_count
                out_tok = resp.usage_metadata.candidates_token_count
                # gemini-2.5-flash: $0.30 / 1M input, $2.50 / 1M output
                cost = (in_tok * 0.30 + out_tok * 2.50) / 1_000_000
                gen.update(output=answer,
                           usage_details={"input_tokens": in_tok, "output_tokens": out_tok})
            except Exception as e:
                answer, cost, error = f"Error: {e}", 0.0, True
                gen.update(output=answer, level="ERROR", status_message=str(e))
            latency = (time.time() - start) * 1000
            trace_id = langfuse.get_current_trace_id()
            langfuse.update_current_trace(user_id=user_id, session_id=session_id)

        # Auto-score quality on the SAME trace (outside the LLM span is fine).
        quality = self.scorer.score_llm_judge(trace_id, message, answer) if not error else 0.0

        self.dashboard.record({"model": MODEL, "latency_ms": latency, "cost_usd": cost,
                               "quality_score": quality, "error": error, "user_id": user_id})
        if quality and quality < 0.5:
            print(f"  ALERT low quality {quality:.2f} for '{message[:40]}'")
        return {"answer": answer, "latency_ms": latency, "cost": cost,
                "quality": quality, "trace_id": trace_id}

# -- Run a small workload --
chat = InstrumentedChat()
for q, user in [("What is RAG?", "student_001"),
                ("Explain transformers", "student_002"),
                ("How do embeddings work?", "student_001")]:
    r = chat.chat(q, user_id=user)
    print(f"  [{user}] {q}  ->  {r['latency_ms']:.0f}ms  ${r['cost']:.5f}  q={r['quality']:.2f}")
chat.dashboard.report()
langfuse.flush()

# Output:
#   [student_001] What is RAG?  ->  980ms  $0.00021  q=0.90
#   [student_002] Explain transformers  ->  1240ms  $0.00034  q=0.86
#   [student_001] How do embeddings work?  ->  1100ms  $0.00029  q=0.88
#   ...dashboard prints the 5 metrics; every call is now visible in Langfuse,
#   auto-scored, per-user, and alertable. This is production-grade observability.


#### 💡 What just happened

What just happened?
  We built a fully instrumented chat application where every single LLM call is: (1) **Traced** in Langfuse with input, output, tokens, and cost. (2) **Auto-scored** for quality using LLM-as-Judge. (3) **Monitored** with local dashboard metrics (latency P50/P95/P99, cost, quality trends). (4) **Alerted** when quality drops below threshold. Each response includes a `trace_id` that links to the full trace in the Langfuse UI. **This is how production AI teams operate: every call is visible, measurable, and debuggable.**

## Recap

> ℹ️ **Info**
>
> What we built
> - **Langfuse manual + decorator tracing** - two approaches: explicit trace/generation creation, or just `@observe()` on any function. Both produce nested trace timelines in the dashboard.
> - **LangSmith tracing** - `@traceable` decorator, auto-traces any function. Deep LangChain integration.
> - **Quality scoring** - 3 methods: user feedback (thumbs up/down), LLM judge (automatic 0-1 score), groundedness check (catches hallucinations). All attached to traces.
> - **ObservabilityDashboard** - 5 key metrics: volume + error rate, latency percentiles, cost breakdown, quality trends, per-model analytics.
> - **InstrumentedChat** - every call traced + auto-scored + monitored + alerted. Trace IDs link to full details. Production-grade observability in one class.
> - **The ecosystem** - tools by architecture (proxy / decorator / OTEL), eval frameworks (RAGAS, DeepEval), burn-rate alerting, silent-drift detection, and a DPDP-compliant India stack.

> ✅ **Info**
>
> 🔗 Where this goes next
> - **This closes Module 7.** You can now build, stream, cost-engineer, chat, tool-call, AND observe a production LLM app.
> - **In Module 14 (LLMOps)** we'll come back to step 7's evaluation and promote it into a full discipline - error analysis and eval-driven development, where the golden dataset drives every change.
> - **In Module 11 (AI Agents)**, this same tracing builds on the trace tree to debug multi-step agent loops - it is how you see what an agent was thinking.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Observability - See Everything Your AI Does**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-7_6.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-7.6.html` - regenerate this notebook whenever the source HTML is updated.*
